# 07 · Frozen proxy and scaling evidence

Compute numerical proxy coefficients **only** for approved country/product/year/pathway decisions. The input decision register must originate in the accepted original research; the notebook will not discover or invent missing B2/D1/D2/D3 reference choices.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

REFERENCE=DATA/'isimip'/'processed'/'07_GLOBAL_LPJmL_CWR_coefficients_2000_2020.csv'
DECISIONS=DATA/'crop'/'evidence'/'approved-proxy-decisions.csv'
OUT=RUNS/'proxy-evidence'/'calculated-proxy-values.csv'
# Required decision columns: target_crop, country_iso3, year, pathway,
# reference_crop, scaling_factor, method_group, evidence_id, status.

In [ ]:
def calculate_frozen_proxies(ref,decisions):
    required=['target_crop','country_iso3','year','pathway','reference_crop',
              'scaling_factor','method_group','evidence_id','status']
    missing=set(required)-set(decisions)
    if missing:raise ValueError(f'Missing approved decision fields: {missing}')
    if decisions[required].isna().any().any():raise ValueError('No arbitrary proxy filling')
    keys=['country_iso3','year','pathway','target_crop']
    if decisions.duplicated(keys).any():raise ValueError('Ambiguous accepted crop route')
    if (pd.to_numeric(decisions.scaling_factor,errors='raise')<0).any():
        raise ValueError('Negative source scaling factor')
    # The accepted reference table may use climate_scenario and fable_product.
    lookup=ref.rename(columns={'climate_scenario':'pathway',
        'fable_product':'reference_crop','fable_product_norm':'reference_crop',
        'median_pirrww_mm_yr':'reference_mm_yr'})
    needed=['country_iso3','year','pathway','reference_crop','reference_mm_yr']
    if set(needed)-set(lookup):raise ValueError('LPJmL reference columns incomplete')
    historical=lookup[lookup.pathway.eq('HISTORICAL')].copy()
    if not historical.empty:
        historical=historical.loc[historical.year.isin([2000,2005])]
        historical=historical.drop(columns='pathway').assign(join_pathway='HISTORICAL')
        historical=pd.concat([historical.assign(pathway=p) for p in ['RCP26','RCP60']],ignore_index=True).drop(columns='join_pathway')
        lookup=pd.concat([lookup[~lookup.pathway.eq('HISTORICAL')],historical],ignore_index=True)
    keyref=['country_iso3','year','pathway','reference_crop']
    if lookup.duplicated(keyref).any():raise ValueError('Nonunique LPJmL reference coefficient')
    merged=decisions.merge(lookup[needed],on=keyref,how='left',validate='many_to_one',indicator=True)
    if not merged['_merge'].eq('both').all():
        raise ValueError('Frozen proxy requests are unsupported in LPJmL; missing values not filled')
    merged['calculated_mm_yr']=merged.reference_mm_yr*merged.scaling_factor
    if not np.isfinite(merged.calculated_mm_yr).all():raise ValueError('Nonfinite proxy values')
    return merged.drop(columns='_merge')

In [ ]:
if not REFERENCE.is_file():raise FileNotFoundError(REFERENCE)
if not DECISIONS.is_file():raise FileNotFoundError(
    f'{DECISIONS} — export the accepted decision register from the original master notebook')
calculated=calculate_frozen_proxies(pd.read_csv(REFERENCE),pd.read_csv(DECISIONS))
print(calculated.groupby('method_group').agg(states=('target_crop','size'),
                                             products=('target_crop','nunique')))
OUT.parent.mkdir(parents=True,exist_ok=True)
if OUT.exists():raise FileExistsError(OUT)
calculated.to_csv(OUT,index=False)